# Long Term Memory in LangGraph

So far, our chatbot could remember within a conversation (short term memory via `InMemorySaver`). 
But start a new thread — and it forgets everything.

**Long term memory** solves this. Using `InMemoryStore`, we can persist facts about a user 
across multiple conversations and threads.

| | Short Term | Long Term |
|---|---|---|
| Tool | `InMemorySaver` | `InMemoryStore` |
| Scope | One thread | All threads |
| Lost when | Program stops | Never (or until cleared) |

In LangGraph, memory store is implemente dusing a baseclass `MemoryStore` with multiple implementations:
- It can create & update memories
- It can retrieve/search memories based on a query
- It can delete memories

in-memory implementation: `InMemoryStore`
database implementation: `SqliteStore` (or `PostgresStore`)
redis implementation: `RedisStore` (for super fast access, but more complex to set up)

##### Namespaces

```
Google Drive
└── 📁 users          ← namespace level 1
    └── 📁 u1         ← namespace level 2  
        └── 📄 name   ← key
        └── 📄 prefs  ← key
```
Namespace = the folder path to organize your data. Without them everything goes into one flat pile — messy. Namespaces let you organize by user, by topic, by anything:
```
("users", "u1")              # user 1's data
("users", "u2")              # user 2's data — completely separate
("users", "u1", "prefs")     # user 1's preferences specifically
```
So two users never mix up each other's memories. So in conclusion namespaces are a way to organise memory in a memory store. They are like folders/directories in a file system, allowing you to group related memories together and avoid key collisions. You can have as many levels of namespaces as you want, and they can be used to structure your data in a way that makes sense for your application.

In [4]:
from langgraph.store.memory import InMemoryStore

store = InMemoryStore()
namespace = ("user", "id")   # folder path for user 1

# CREATE — store.put()
store.put(namespace, "1", {"data": "User likes pizza"})
store.put(namespace, "2", {"data": "User prefers dark mode"})

In [ ]:
# READ — store.get()
item = store.get(namespace, "2")

print(item.value)      # {'data': 'User prefers dark mode'}
print(item.key)        # '2'
print(item.namespace)  # ['user', 'id']

{'data': 'User prefers dark mode'}
2
('user', 'id')


In [9]:
# READ ALL — store.search()
pythonitems = store.search(namespace)   # get everything in this namespace

for item in pythonitems:
    print(item.value)

{'data': 'User likes pizza'}
{'data': 'User prefers dark mode'}


## Semantic Search in InMemoryStore

So far, `store.search(namespace)` returned **all memories** — no filtering.

But what if a user has 100 stored memories? Dumping all of them into the LLM prompt is expensive and noisy.

**Semantic search** lets us retrieve only the most *relevant* memories based on meaning — not exact keyword match.
```python
store.search(namespace, query="what is the user currently learning", limit=1)
# → "User is learning machine learning" ✅  
# → ignores irrelevant memories like "User likes pizza" ✅
```

To enable this, we attach an **embedding model** to the store. It converts text into vectors so similarity can be measured mathematically.
```python
store = InMemoryStore(index={
    'embed': embedding_model,   # converts text → vectors
    'dims': 1536                # vector size for this model
})
```

Now every `store.put()` auto-embeds the value, and `store.search(query=...)` finds the closest matches.